← [09 - Enigme d'Einstein](Z3-09-Enigme-Einstein-Python.ipynb) | [README Z3-Python](README.md) | [11 - Coloration de graphe >>](Z3-11-Graph-Coloring-Python.ipynb)
# 10. Cryptarithmes (SEND + MORE = MONEY)
Un **cryptarithme** est une equation arithmetique ou chaque lettre represente un **chiffre distinct** (0-9), et ou les mots formant l'equation correspondent a des nombres en ecriture positionnelle. Le classique `SEND + MORE = MONEY` admet une unique solution : laquelle ?
Ce type de puzzle est un cas canonique de **satisfaction de contraintes sur les entiers** : 8 lettres, chacune dans `{0..9}`, deux a deux distinctes, avec des tetes non nulles (S et M) et une equation lineaire positionnelle. La **propagation de contraintes** du solveur SMT resout instantanement ce que la brute force met plusieurs centaines de millisecondes a enumerer.
> Port pyz3 du notebook C# [`13_Cryptarithmetic_SMT.ipynb`](../Z3-Linq2Z3/13_Cryptarithmetic_SMT.ipynb) (Microsoft.Z3 brut).

In [1]:
from z3 import *
import time
print("Imports OK : z3-solver (pyz3)")


Imports OK : z3-solver (pyz3)


## Modelisation : SEND + MORE = MONEY

On encode chaque lettre comme un entier `Int` dans `{0..9}`, on impose la **distinction globale** (`Distinct`), les têtes non nulles (`S > 0`, `M > 0`), puis l'**equation positionnelle** :

```
  1000*S + 100*E + 10*N + D     (SEND)
+ 1000*M + 100*O + 10*R + E     (MORE)
= 10000*M + 1000*O + 100*N + 10*E + Y   (MONEY)
```

Z3 propage ces contraintes et renvoie l'unique modèle.

In [2]:
# Cryptarithme classique : SEND + MORE = MONEY
s = Solver()
S, E, N, D, M, O, R, Y = Ints('S E N D M O R Y')
letters = [S, E, N, D, M, O, R, Y]

# (1) Domaine 0..9 pour chaque lettre
s.add([And(v >= 0, v <= 9) for v in letters])

# (2) Les 8 lettres sont deux a deux distinctes (contrainte globale)
s.add(Distinct(letters))

# (3) Tetes non nulles (pas de zero non-significatif)
s.add(S > 0, M > 0)

# (4) Equation arithmetique positionnelle : SEND + MORE == MONEY
s.add((1000*S + 100*E + 10*N + D) +
      (1000*M + 100*O + 10*R + E) ==
      (10000*M + 1000*O + 100*N + 10*E + Y))

if s.check() != sat:
    print("UNSAT : aucune solution (inattendu pour ce cryptarithme)")
else:
    m = s.model()
    send  = 1000*m[S].as_long() + 100*m[E].as_long() + 10*m[N].as_long() + m[D].as_long()
    more  = 1000*m[M].as_long() + 100*m[O].as_long() + 10*m[R].as_long() + m[E].as_long()
    money = 10000*m[M].as_long() + 1000*m[O].as_long() + 100*m[N].as_long() + 10*m[E].as_long() + m[Y].as_long()
    print("  S=%d E=%d N=%d D=%d  M=%d O=%d R=%d Y=%d" % (m[S].as_long(), m[E].as_long(), m[N].as_long(), m[D].as_long(),
                                                            m[M].as_long(), m[O].as_long(), m[R].as_long(), m[Y].as_long()))
    print("    %5d" % send)
    print("  + %5d" % more)
    print("  -------")
    print("  %5d    (%d + %d = %d, verifie = %s)" % (money, send, more, money, send + more == money))


  S=9 E=5 N=6 D=7  M=1 O=0 R=8 Y=2
     9567
  +  1085
  -------
  10652    (9567 + 1085 = 10652, verifie = True)


## Interpretation

La solution unique est `S=9 E=5 N=6 D=7 M=1 O=0 R=8 Y=2`, soit `9567 + 1085 = 10652`. Notez le rôle de la **retenue** : ce n'est pas evident a l'oeil, c'est la propagation arithmetique du solveur qui la deduit automatiquement a partir de l'equation positionnelle. Aucune boucle explicite sur les retenues n'est necessaire : c'est précisément ce que Z3 fait mieux qu'un code imperatif.

## Anatomie : où sont passées les retenues ?

L'encodage ci-dessus n'introduit **aucune variable de retenue** — et pourtant l'addition posée à la main en exige. Reconstituons la chaîne des retenues que la propagation de Z3 retrouve implicitement, en notant $c_i$ la retenue sortant de la colonne $i$ (de droite à gauche) :

| Colonne | Équation manuelle | Avec la solution mesurée |
|---|---|---|
| unités | $D + E = 10 c_1 + Y$ | $7 + 5 = 12$, donc $c_1 = 1$, $Y = 2$ |
| dizaines | $N + R + c_1 = 10 c_2 + E$ | $6 + 8 + 1 = 15$, donc $c_2 = 1$, $E = 5$ |
| centaines | $E + O + c_2 = 10 c_3 + N$ | $5 + 0 + 1 = 6$, donc $c_3 = 0$, $N = 6$ |
| milliers | $S + M + c_3 = 10 c_4 + O$ | $9 + 1 + 0 = 10$, donc $c_4 = 1$, $O = 0$ |
| dizaines de milliers | $c_4 = M$ | $1 = M$ |

Toute cette chaîne est **subsumée** par l'unique équation linéaire positionnelle : l'arithmétique entière de Z3 raisonne sur l'égalité $9567 + 1085 = 10652$ comme sur n'importe quelle identité entière, sans qu'on ait à décomposer les colonnes. Le chiffre de tête $M = 1$ n'est rien d'autre que la dernière retenue devenue chiffre significatif — le solveur la déduit, il ne la déclare pas.

## Approche naive vs solveur : que fait la propagation ?

La brute force enumere tous les **arrangements** `P(10, 8) = 10!/(10-8)! = 1 814 400` candidats (8 lettres distinctes prises parmi 10 chiffres), puis teste l'equation pour chacun. Le solveur, lui, **propage** les contraintes (distinction, domaines, equation) pour eliminer en bloc des familles entieres d'affectations impossibles avant même de les tester. Mesurons l'ecart.

In [3]:
# (1) Brute force : 8 boucles imbriquees, on saute des que deux lettres coincident
#     P(10,8) = 1 814 400 candidats distincts testes.
nb_brute = 0
t0 = time.perf_counter()
for S in range(10):
 for E in range(10):
  if E == S: continue
  for N in range(10):
   if N in (S, E): continue
   for D in range(10):
    if D in (S, E, N): continue
    for M in range(10):
     if M in (S, E, N, D): continue
     for O in range(10):
      if O in (S, E, N, D, M): continue
      for R in range(10):
       if R in (S, E, N, D, M, O): continue
       for Y in range(10):
        if Y in (S, E, N, D, M, O, R): continue
        if S > 0 and M > 0:
            send  = 1000*S + 100*E + 10*N + D
            more  = 1000*M + 100*O + 10*R + E
            money = 10000*M + 1000*O + 100*N + 10*E + Y
            if send + more == money:
                nb_brute += 1
t_brute = time.perf_counter() - t0

# (2) Z3 : meme theoreme que la cellule precedente, mesure
t0 = time.perf_counter()
nb_z3 = 0
s2 = Solver()
S2, E2, N2, D2, M2, O2, R2, Y2 = Ints('S2 E2 N2 D2 M2 O2 R2 Y2')
L2 = [S2, E2, N2, D2, M2, O2, R2, Y2]
s2.add([And(v >= 0, v <= 9) for v in L2])
s2.add(Distinct(L2))
s2.add(S2 > 0, M2 > 0)
s2.add((1000*S2 + 100*E2 + 10*N2 + D2) + (1000*M2 + 100*O2 + 10*R2 + E2) ==
       (10000*M2 + 1000*O2 + 100*N2 + 10*E2 + Y2))
if s2.check() == sat:
    nb_z3 = 1
t_z3 = time.perf_counter() - t0

print("Brute force : %d solution(s) en %.1f ms  (1 814 400 candidats testes)" % (nb_brute, t_brute * 1000))
print("Z3 solve    : %d solution(s) en %.1f ms  (propagation de contraintes)" % (nb_z3, t_z3 * 1000))
print(" -> meme resultat (%s), strategies differentes." % (nb_brute == nb_z3))


Brute force : 1 solution(s) en 2156.4 ms  (1 814 400 candidats testes)
Z3 solve    : 1 solution(s) en 15.4 ms  (propagation de contraintes)
 -> meme resultat (True), strategies differentes.


## Lecture du benchmark

L'ecart est net : la brute force teste **1 814 400** affectations pour trouver l'unique solution, tandis que Z3 la trouve en quelques millisecondes via propagation. Ce n'est pas une question de vitesse brute du langage : c'est la **nature de la recherche**. Le solveur ne << teste >> pas, il **deduit** - les contraintes `Distinct` et l'equation positionnelle reduisent l'espace avant tout branchement. C'est l'intérêt pedagogique central : sur un problème combinatoire, declarer les contraintes bat l'enumeration imperative.

## Témoin et preuve : deux niveaux de garantie

`check()` renvoie `sat` et un modèle : c'est un **témoin** de l'existence d'au moins une solution, pas une preuve d'unicité. L'exemple guidé 2 montre comment bloquer les modèles trouvés et chercher à établir leur exhaustivité. Une réponse `unknown` ne prouve pas l'absence d'autres modèles.

## Variante : CROSS + ROADS = DANGER (9 lettres distinctes)

On complique : 9 lettres distinctes (`C, R, O, S, A, D, N, G, E`), trois têtes non nulles (`C`, `R`, `D`), et une equation a 6 chiffres en résultats. La brute force devrait enumerer `P(10, 9) = 3 628 800` candidats - le solveur reste instantane.

In [4]:
# Variante : CROSS + ROADS = DANGER (9 lettres distinctes)
s3 = Solver()
C, R, O, S, A, D, N, G, E = Ints('C R O S A D N G E')
L3 = [C, R, O, S, A, D, N, G, E]

# Domaine 0..9 pour les 9 lettres
s3.add([And(v >= 0, v <= 9) for v in L3])

# 9 lettres deux a deux distinctes
s3.add(Distinct(L3))

# Tetes non nulles : C (CROSS), R (ROADS), D (DANGER)
s3.add(C > 0, R > 0, D > 0)

# CROSS + ROADS == DANGER
# CROSS  = 10000*C + 1000*R + 100*O + 10*S + S
# ROADS  = 10000*R + 1000*O + 100*A + 10*D + S
# DANGER = 100000*D + 10000*A + 1000*N + 100*G + 10*E + R
s3.add((10000*C + 1000*R + 100*O + 10*S + S) +
       (10000*R + 1000*O + 100*A + 10*D + S) ==
       (100000*D + 10000*A + 1000*N + 100*G + 10*E + R))

if s3.check() != sat:
    print("UNSAT : aucune solution pour CROSS + ROADS = DANGER")
else:
    m = s3.model()
    cross  = 10000*m[C].as_long() + 1000*m[R].as_long() + 100*m[O].as_long() + 10*m[S].as_long() + m[S].as_long()
    roads  = 10000*m[R].as_long() + 1000*m[O].as_long() + 100*m[A].as_long() + 10*m[D].as_long() + m[S].as_long()
    danger = 100000*m[D].as_long() + 10000*m[A].as_long() + 1000*m[N].as_long() + 100*m[G].as_long() + 10*m[E].as_long() + m[R].as_long()
    print("CROSS=%d  ROADS=%d  DANGER=%d   (%d + %d = %d, egal a DANGER = %d : %s)" %
          (cross, roads, danger, cross, roads, cross + roads, danger, cross + roads == danger))


CROSS=96233  ROADS=62513  DANGER=158746   (96233 + 62513 = 158746, egal a DANGER = 158746 : True)


## Lecture de la variante CROSS + ROADS = DANGER

La sortie présente un modèle : vérifiez que les deux opérandes donnent bien le résultat affiché. La somme de deux nombres à cinq chiffres reste inférieure à 200 000 : la tête `D` du résultat à six chiffres vaut donc nécessairement 1. La lettre `S` apparaît deux fois dans CROSS et une fois dans ROADS : ces occurrences partagent une seule variable. Neuf lettres distinctes occupent neuf chiffres ; le modèle affiché ne suffit pas à établir son unicité.

## Exemples guidés

Les trois réalisations suivantes de Léane Audegond (EPF), issues du deuxième TP, illustrent la modélisation d'une addition, l'énumération des modèles et une multiplication non linéaire. Les exercices non résolus viennent ensuite.

### Exemple guidé 1 — DONALD + GERALD = ROBERT

Réalisation de Léane Audegond (EPF), issue du deuxième TP.

On modélise dix lettres distinctes, les chiffres de tête non nuls et l'égalité positionnelle :

- `DONALD = 100000*D + 10000*O + 1000*N + 100*A + 10*L + D`
- `GERALD = 100000*G + 10000*E + 1000*R + 100*A + 10*L + D`
- `ROBERT = 100000*R + 10000*O + 1000*B + 100*E + 10*R + T`

Dans DONALD, D apparaît en tête et aux unités ; dans ROBERT, R apparaît en tête et aux dizaines.

In [5]:
# Exercice 1 : DONALD + GERALD = ROBERT
# TODO etudiant :
# Etape 1 : D, O, N, A, L, G, E, R, B, T = Ints('D O N A L G E R B T')
# Etape 2 : domaine 0..9 pour les 10 lettres
# Etape 3 : Distinct(...) sur les 10 lettres
# Etape 4 : têtes non nulles (D > 0, G > 0, R > 0)
# Etape 5 : equation DONALD + GERALD == ROBERT
#result = None  # TODO etudiant
#print("Exercice 1 a completer : modelisez DONALD + GERALD = ROBERT avec Z3")


from z3 import *
def exercice_1():
    
    D, O, N, A, L, G, E, R, B, T = Ints('D O N A L G E R B T') #on créé 10 inconnues entières (int)
    lettres = [D, O, N, A, L, G, E, R, B, T] #regorupe les variables dans une seule liste pour ne pas les réécrire plus tard à chaque fois

    s = Solver()   # le solveur qui va recevoir les règles

    # chaque lettre est un chiffre entre 0 et 9
    s.add([And(v >= 0, v <= 9) for v in lettres])

    # deux lettres différentes = deux chiffres différents
    s.add(Distinct(lettres))

    # la première lettre de chaque mot n'est pas 0
    s.add(D > 0, G > 0, R > 0)

    # l'addition posée en colonnes.
    # chaque lettre est multipliée par son rang (unités, dizaines, centaines...)
    donald = 100000*D + 10000*O + 1000*N + 100*A + 10*L + D  
    gerald = 100000*G + 10000*E + 1000*R + 100*A + 10*L + D
    robert = 100000*R + 10000*O + 1000*B + 100*E + 10*R + T
    s.add(donald + gerald == robert)

    # On lance la recherche
    # on va demander s'il y a une solution avec une boucle if 
    #on regarde d'abord s'il y en a une avec s.check() == sat
    if s.check() == sat: 
        m = s.model() #si oui on recupere le model et on l'affiche
        v = {str(x): m[x].as_long() for x in lettres}   # dictionnaire qui associe chaque lettre à son chiffre
        print("Solution :", v)  #affiche la solution trouvée

        # On reconstruit les 3 nombres pour vérifier si c'est correct
        #on va donc chercher dans le dictionnaire les valeurs pour chaque lettre, les transformer en texte et les assembler 
        n_donald = int("".join(str(v[c]) for c in "DONALD"))
        n_gerald = int("".join(str(v[c]) for c in "GERALD"))
        n_robert = int("".join(str(v[c]) for c in "ROBERT"))
        print(f"{n_donald} + {n_gerald} = {n_robert}") #ici on affiche le resultat de l'addition pour verifier que c'est correct 
        
    else: # si aucune solution n'est trouvée, on affiche un message
        print("Aucune solution")

exercice_1()
print()

Solution : {'D': 5, 'O': 2, 'N': 6, 'A': 4, 'L': 8, 'G': 1, 'E': 9, 'R': 7, 'B': 3, 'T': 0}
526485 + 197485 = 723970



### Exemple guidé 2 — Énumérer les solutions de SEND + MORE = MONEY

Réalisation de Léane Audegond (EPF), issue du deuxième TP. Après chaque modèle, une disjonction impose qu'au moins une lettre change. Quand la recherche se termine par `unsat`, tous les modèles ont été énumérés ; un seul modèle établit alors l'unicité.

In [6]:
# Exercice 2 : prouver l'unicite de SEND + MORE = MONEY
# TODO etudiant :
# 1. Re-resoudre, recuperer le modele m
# 2. Ajouter Or(S != m[S], E != m[E], N != m[N], ...) pour nier la solution
# 3. Si unsat -> unique ; sinon compter toutes les solutions par iteration
#nb_solutions = None  # TODO etudiant
#print("Exercice 2 a completer : prouvez l'unicite (ou enumerez toutes les solutions)")


def exercice_2():
    S, E, N, D, M, O, R, Y = Ints('S E N D M O R Y')
    lettres = [S, E, N, D, M, O, R, Y]

    s = Solver()
    s.add([And(v >= 0, v <= 9) for v in lettres])    # chiffres 0..9
    s.add(Distinct(lettres))                         # tous différents
    s.add(S > 0, M > 0)                              # 1eres lettres non nulles
    s.add((1000*S + 100*E + 10*N + D) +
          (1000*M + 100*O + 10*R + E) ==
          (10000*M + 1000*O + 100*N + 10*E + Y))     # l'addition


    nb_solutions = 0 #initie le compteur de solutions

    # Boucle while tant que Z3 trouve une solution, on la note,
    # puis on lui interdit cette solution et on redemande

    while s.check() == sat:
        m = s.model() #affiche le model de la solution trouvée

        nb_solutions += 1 #fait monter le compteur de solutions à chaque nouvelle solutions trouvée

        #on affiche le numéro de la solution et le dictionnaire qui associe chaque lettre à son chiffre
        print("Solution", nb_solutions, ":", {str(v): m[v].as_long() for v in lettres}) 

        # on va ajouter une nouvelle contrainte pour interdire la solution que l'on vient de trouver
        # et forcer Z3 à chercher une autre solution
        # Test si au moins une lettre est différente de la solution trouvée avant
        s.add(Or([v != m[v] for v in lettres]))

    # Quand check() répond enfin unsat, il n'y en a plus d'autre
    print("Nombre total de solutions :", nb_solutions)

    if nb_solutions == 1: #affiche le nombre de solutions trouvées, et si c'est 1, on affiche que la solution est unique
        print("-> La solution est unique")
    return nb_solutions

exercice_2()
print()


Solution 1 : {'S': 9, 'E': 5, 'N': 6, 'D': 7, 'M': 1, 'O': 0, 'R': 8, 'Y': 2}
Nombre total de solutions : 1
-> La solution est unique



### Exemple guidé 3 — Une multiplication sans solution

Réalisation de Léane Audegond (EPF), issue du deuxième TP. On cherche `ABC * D = DBC` en base dix, avec quatre chiffres distincts et `A, D > 0`. Contrairement à une multiplication par une constante, le produit par `D` est non linéaire. Le résultat `unsat` concerne précisément ces contraintes ; il ne démontre pas l'impossibilité d'autres multiplications.

In [7]:
# Exercice 3 : ABC * D = DBC (multiplication)
# TODO etudiant :
# 1. A, B, C, D = Ints('A B C D'), domaine 0..9
# 2. Distinct(A, B, C, D), têtes A > 0 et D > 0
# 3. Equation : (100*A + 10*B + C) * D == (100*D + 10*B + C)
#result = None  # TODO etudiant
#print("Exercice 3 a completer : modelisez ABC * D = DBC")

def exercice_3():
    A, B, C, D = Ints('A B C D')   # on crée les 4 variables
    lettres = [A, B, C, D]         # on les range dans une liste

    s = Solver()
    s.add([And(v >= 0, v <= 9) for v in lettres])   # chiffres entre 0 et 9
    s.add(Distinct(lettres)) # 4 chiffres tous différents
    s.add(A > 0, D > 0) # 1ere lettres non nulles
    
    s.add((100*A + 10*B + C) * D == (100*D + 10*B + C))   # ABC * D = DBC

    if s.check() == sat: #boucle if pour savoir si une solution est trouvée
        m = s.model() #affiche le model de la solution trouvée
        print("Solution :", {str(v): m[v].as_long() for v in lettres}) #print le dictionnaire qui associe chaque lettre à son chiffre
    else:
        # unsat : l'équation force A = 1, puis soit D = 1, soit B = C = 0,
        # ce qui est impossible si les 4 chiffres doivent être différents
        print("unsat : aucune solution avec 4 chiffres distincts") #si aucune solution n'est trouvée, on affiche un message


exercice_3()
print()

unsat : aucune solution avec 4 chiffres distincts



## Exercices à compléter

### Exercice 1 — Retenues et équivalence

Pour `SEND + MORE = MONEY`, introduisez quatre retenues entières comprises entre 0 et 1 et écrivez les équations de chaque colonne. Sous les mêmes domaines, distinctions et contraintes de tête que le modèle positionnel, demandez à Z3 un contre-exemple dans chaque sens entre les deux encodages. Précisez la portée de la comparaison : les retenues sont des variables auxiliaires à éliminer existentiellement. Interprétez les réponses `sat`, `unsat` ou `unknown`.

In [8]:
# TODO : définir les retenues, les équations et les deux recherches de contre-exemple.
print("Exercice 1 à compléter : comparer les encodages avec et sans retenues")

Exercice 1 à compléter : comparer les encodages avec et sans retenues


### Exercice 2 — Changer de base

Pour chaque base entière fixée de 4 à 10 incluse, modélisez `AB + BA = CC` : les lettres désignent trois chiffres distincts de la base et A, B et C sont non nuls. Utilisez les poids positionnels propres à la base, puis affichez pour chaque base le statut et, lorsqu'il existe, un modèle vérifié par substitution. Distinguez absence de solution et réponse `unknown`.

In [9]:
# TODO : construire un solveur par base et vérifier chaque modèle trouvé.
print("Exercice 2 à compléter : cryptarithme en bases variables")

Exercice 2 à compléter : cryptarithme en bases variables


### Exercice 3 — Expliquer une contradiction

Reprenez `ABC * D = DBC` avec les domaines décimaux, les têtes non nulles et la distinction des lettres. Associez des étiquettes distinctes aux groupes de contraintes avec `assert_and_track`, demandez un noyau insatisfiable, puis vérifiez que les contraintes correspondant aux étiquettes retournées sont elles-mêmes insatisfiables. Expliquez pourquoi un noyau n'est pas nécessairement minimal et distinguez `unknown` de `unsat`.

In [10]:
# TODO : suivre les contraintes, extraire le noyau et vérifier sa contradiction.
print("Exercice 3 à compléter : noyau insatisfiable")

Exercice 3 à compléter : noyau insatisfiable


### Exercice 4 — Cryptarithme avec produit (multiplication)
Généralisez à la **multiplication** : `ABCD * 4 = DCBA`, où `ABCD = 1000*A + 100*B + 10*C + D` et `DCBA = 1000*D + 100*C + 10*B + A`.

**Étapes** :
1. Variables `A, B, C, D = Ints('A B C D')`, domaine `{0..9}`.
2. `Distinct(A, B, C, D)`, têtes `A > 0` et `D > 0`.
3. Équation positionnelle : `(1000*A + 100*B + 10*C + D) * 4 == (1000*D + 100*C + 10*B + A)`.
4. Résolvez, puis **prouvez l'unicité** comme dans l'exemple guidé 2 : ajoutez la négation du modèle trouvé et vérifiez que le solveur répond `unsat`.

**Indice** : contrairement à `SEND + MORE = MONEY`, le multiplicateur n'est pas une variable mais la **constante 4**. Commencez par le chiffre des unités : `A` est le dernier chiffre de `D * 4`, et `D` est le premier chiffre du produit — les deux se contraignent mutuellement.

In [11]:
# Exercice 4 : ABCD * 4 = DCBA (multiplication)
# TODO etudiant :
# 1. A, B, C, D = Ints('A B C D'), domaine 0..9
# 2. Distinct(A, B, C, D), têtes A > 0 et D > 0
# 3. Equation : (1000*A + 100*B + 10*C + D) * 4 == (1000*D + 100*C + 10*B + A)
# 4. Unicite : ajouter la negation du modele trouve et verifier unsat
result = None  # TODO etudiant
print("Exercice 4 a completer : modelisez ABCD * 4 = DCBA")

Exercice 4 a completer : modelisez ABCD * 4 = DCBA


## Conclusion

Les cryptarithmes illustrent le paradigme déclaratif : on exprime les domaines, les contraintes de distinction et l'équation plutôt que l'algorithme de recherche. Les exemples montrent la recherche d'un témoin, l'énumération des modèles et l'insatisfiabilité d'une multiplication. Les exercices prolongent ces méthodes avec des retenues explicites, des bases variables et un noyau insatisfiable.

Le pont avec la série sœur Z3.Linq (C#) se fait via les exemples portés entre Microsoft.Z3 et pyz3.

***

> Carnet issu de l'EPIC #1206.
